<a href="https://colab.research.google.com/github/Selvapriya05/Selvapriya-Codeboosters-2026/blob/main/Day_8/Day_8_Miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers pandas groq

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving college_notes.csv to college_notes (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("college_notes.csv")
print(df.head())
print(df.columns)

  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  
Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="college_notes")

In [ ]:
for i, row in df.iterrows():
    text = row["content"]

    embedding = embedding_model.encode(text).tolist()

    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[text]
    )

print("All notes indexed successfully!")

All notes indexed successfully!


In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    return results["documents"][0]

In [ ]:
from groq import Groq

client = Groq(api_key="GROQ_API_KEY")

In [ ]:
def answer_question(question):
    chunks = retrieve_relevant_chunks(question)

    context = "\n\n".join(chunks)

    prompt = f"""
You are a helpful AI tutor.

Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and simply:
"""

    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
question = input("Ask your question: ")

answer = answer_question(question)
print("\n Answer:\n", answer)